# Fill a missing AdmR value: one drug

This is the first, deliberately small run of deliverable D3: fill a `NaN` in the `AdmR` column of `inputs/mode_of_administration.md` for **one drug and one ATC code**, using DailyMed label counts.

Rules applied (from the task spec):

- **Rule 3**: at most 2 routes per ATC code, intravenous excluded.
- **Rule 4**: rank routes by how many DailyMed labels report them for this ATC code.
- **Rule 5**: one output row per route.
- **Rule 7**: every label used is saved to `results/` with its DailyMed link and a timestamp; the raw API response is kept in `evidence/dailymed/`.

The source table is never edited in place; results go to a new CSV.

In [1]:
import json, re, datetime
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

DRUG = "azelastine"
ATC = "S01GX07"
# Label for eye-drop routes. WHO DDD codes have none; "ophthalmic" is the default until Star decides.
OPHTHALMIC_LABEL = "ophthalmic"

HERE = Path.cwd() if (Path.cwd() / "inputs").exists() else Path.cwd() / "curation"
INPUT = HERE / "inputs" / "mode_of_administration.md"
CACHE = HERE / "evidence" / "dailymed" / f"{DRUG}_spls.json"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

## 1. Load the table and find the row

In [2]:
rows = [l.strip().strip("|").split(" | ") for l in INPUT.read_text().splitlines() if l.startswith("|")]
rows = [[c.strip() for c in r] for r in rows if len(r) >= 4]
admr = pd.DataFrame(rows[2:], columns=rows[0])
print(len(admr), "rows,", (admr.AdmR == "NaN").sum(), "with AdmR = NaN")
admr[admr.GenericName == DRUG]

1167 rows, 315 with AdmR = NaN


,CID,GenericName,ATC,AdmR
11,2267,azelastine,R06AX19,O
12,2267,azelastine,S01GX07,NaN
13,2267,azelastine,R01AC03,N


The existing `AdmR` codes are WHO ATC/DDD route codes (O, P, N, R, TD, SL, ...). WHO assigns no DDD, and so no route, to most eye (S01), anaesthetic (N01) and skin (D) products, which is why so many of these rows are empty.

In [3]:
admr.AdmR.value_counts().head(12)

AdmR
O                 477
NaN               315
P                 237
R                  26
N                  22
Inhal.powder       19
Inhal.solution     13
Inhal.aerosol      13
TD                 11
V                  10
SL                 10
implant             4
Name: count, dtype: int64

## 2. Get the DailyMed labels

Live call to the DailyMed SPL API. If the network blocks it, the notebook falls back to the saved response in `evidence/dailymed/`, so the run is reproducible offline.

In [4]:
def dailymed_spls(name):
    url = "https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json"
    out, page = [], 1
    while True:
        r = requests.get(url, params={"drug_name": name, "pagesize": 100, "page": page}, timeout=20)
        r.raise_for_status()
        j = r.json()
        out += [{"setid": d["setid"], "title": d["title"]} for d in j["data"]]
        if j["metadata"]["next_page"] in (None, "null"):
            return out

try:
    labels = dailymed_spls(DRUG)
    retrieved = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    CACHE.write_text(json.dumps({"retrieved_utc": retrieved, "labels": labels}, indent=1))
    source = "live"
except Exception as e:
    cached = json.loads(CACHE.read_text())
    labels, retrieved, source = cached["labels"], cached["retrieved_utc"], f"cache ({type(e).__name__})"
print(len(labels), "labels from", source, "retrieved", retrieved)

47 labels from cache (ProxyError) retrieved 2026-10-09T03:37:06+00:00


## 3. Route of each label, and which ATC code it belongs to

The route is read from the dosage form in the label title. A drug can have several ATC codes (azelastine: R06AX19 oral, R01AC03 nasal, S01GX07 eye), so only labels whose route fits **this** ATC code are counted (rule 4 is per ATC).

In [5]:
FORM_TO_ROUTE = [  # first match wins
    (r"OPHTHALMIC|SOLUTION/ ?DROPS", OPHTHALMIC_LABEL),
    (r"SPRAY", "N"),
    (r"INJECTION|INTRAVENOUS", "IV"),
    (r"TABLET|CAPSULE|SYRUP|ORAL", "O"),
]
ATC_PREFIX_ROUTES = {"S01": {OPHTHALMIC_LABEL}, "R01": {"N"}, "R06": {"O"}}

def route_of(title):
    for pattern, route in FORM_TO_ROUTE:
        if re.search(pattern, title.upper()):
            return route
    return "unknown"

lab = pd.DataFrame(labels)
lab["route"] = lab.title.map(route_of)
allowed = ATC_PREFIX_ROUTES.get(ATC[:3])
lab["counts_for_this_ATC"] = lab.route.isin(allowed) if allowed else True
lab.route.value_counts()

route
N             38
ophthalmic     9
Name: count, dtype: int64

## 4. Apply rules 3 and 4: drop IV, keep the top 2

In [6]:
counts = Counter(lab.loc[lab.counts_for_this_ATC & (lab.route != "IV"), "route"])
top = counts.most_common(2)
print(f"{ATC}: route counts {dict(counts)} -> keep {top}")

S01GX07: route counts {'ophthalmic': 9} -> keep [('ophthalmic', 9)]


## 5. Rule 5: one row per route, then save

In [7]:
n_total = len(lab)
out = pd.DataFrame([{
    "CID": admr.loc[(admr.GenericName == DRUG) & (admr.ATC == ATC), "CID"].iloc[0],
    "GenericName": DRUG, "ATC": ATC, "AdmR_before": "NaN", "AdmR_after": route,
    "Edit_Flag": "filled_from_DailyMed",
    "evidence": f"{n} of {n_total} DailyMed labels for {DRUG} have this route",
} for route, n in top])
out.to_csv(RESULTS / f"admr_{DRUG}_{ATC}.csv", index=False)

prov = lab.assign(drug=DRUG, source_url="https://dailymed.nlm.nih.gov/dailymed/drugInfo.cfm?setid=" + lab.setid,
                  retrieved_utc=retrieved)
prov.to_csv(RESULTS / f"provenance_{DRUG}.csv", index=False)
out

,CID,GenericName,ATC,AdmR_before,AdmR_after,Edit_Flag,evidence
0,2267,azelastine,S01GX07,NaN,ophthalmic,filled_from_DailyMed,9 of 47 DailyMed labels for azelastine have th...


## What is still open

- The label for eye routes (`OPHTHALMIC_LABEL`): "ophthalmic" or a short code, waiting on Star.
- IV-only drugs such as alfentanil (N01AH02) have no route left after rule 3; how to record them is waiting on Star.
- PubChem, WHO ATC/DDD and DrugBank are blocked from the cloud session that ran this notebook, so only DailyMed was used.